# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [59]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [60]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [61]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [62]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [63]:
 # Add a column that creates a unique key to identify each record in order to answer questions about individual trips
from pyspark.sql.functions import monotonically_increasing_id

df_trips = df_trips.withColumn(
    "tripID",
    monotonically_increasing_id()
)

In [64]:
df_trips.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|     tripID|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1

In [65]:
# Which trip has the highest passenger count
df_trips.select("tripId", "passenger_count").orderBy("passenger_count", ascending=False).show(1)

+-----------+---------------+
|     tripId|passenger_count|
+-----------+---------------+
|60130492100|            9.0|
+-----------+---------------+
only showing top 1 row


The trip with the highest passenger count is the trip with ID 60130492100.

In [66]:
# What is the Average passenger count
from pyspark.sql.functions import avg

df_trips.select(avg("passenger_count")).show()

+--------------------+
|avg(passenger_count)|
+--------------------+
|  1.5670317144945614|
+--------------------+



In [67]:
# Shortest/longest trip by distance?
df_trips.select("tripID", "trip_distance") \
    .orderBy("trip_distance", ascending=True) \
    .show(1)


+-----------+-------------+
|     tripID|trip_distance|
+-----------+-------------+
|60129542146|          0.0|
+-----------+-------------+
only showing top 1 row


In [68]:
# Longest trip by distance
df_trips.select("tripID", "trip_distance") \
    .orderBy("trip_distance", ascending=False) \
    .show(1)

+-----------+-------------+
|     tripID|trip_distance|
+-----------+-------------+
|60135616235|        831.8|
+-----------+-------------+
only showing top 1 row


In [69]:
# Create duration
from pyspark.sql.functions import unix_timestamp

df_trips = df_trips.withColumn(
    "trip_duration",
    (unix_timestamp("tpep_dropoff_datetime") -
    unix_timestamp("tpep_pickup_datetime"))/60
)

In [70]:
# Shortest trip by time
df_trips.select("tripID", "trip_duration") \
    .orderBy("trip_duration", ascending=True) \
    .show(1)

+-----------+-------------+
|     tripID|trip_duration|
+-----------+-------------+
|60130745328|     -84280.5|
+-----------+-------------+
only showing top 1 row


In [71]:
# Longest trip by time
df_trips.select("tripID", "trip_duration") \
    .orderBy("trip_duration", ascending=False) \
    .show(1)

+-----------+-----------------+
|     tripID|    trip_duration|
+-----------+-----------------+
|60129610411|43648.01666666667|
+-----------+-----------------+
only showing top 1 row


In [72]:
# busiest day/slowest single day
from pyspark.sql.functions import to_date, count

# Add day column 
df_trips = df_trips.withColumn(
    "day",
    to_date("tpep_pickup_datetime")
)

# Busiest day
df_trips.groupBy("day").count() \
 .orderBy("count", ascending=False) \
 .show(1)


+----------+------+
|       day| count|
+----------+------+
|2019-01-25|292499|
+----------+------+
only showing top 1 row


In [73]:
# Slowest day
df_trips.groupBy("day").count() \
 .orderBy("count", ascending=True) \
 .show(1)

+----------+-----+
|       day|count|
+----------+-----+
|2019-05-20|    1|
+----------+-----+
only showing top 1 row


In [74]:
# busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
from pyspark.sql.functions import hour

# Add hour column
df_trips = df_trips.withColumn(
    "hour",
    hour("tpep_pickup_datetime")
)

# Busiest time of day 
df_trips.groupBy("hour").count() \
    .orderBy("count", ascending=False) \
    .show(1)



+----+------+
|hour| count|
+----+------+
|  18|515390|
+----+------+
only showing top 1 row


In [75]:
# Slowest time of the day
df_trips.groupBy("hour").count() \
    .orderBy("count", ascending=True) \
    .show(1)

+----+-----+
|hour|count|
+----+-----+
|   4|61424|
+----+-----+
only showing top 1 row


In [76]:
# On average which day of the week is slowest/busiest
from pyspark.sql.functions import dayofweek, avg, count

# Add day of the week column
df_trips = df_trips.withColumn(
    "day_of_week",
    dayofweek("tpep_pickup_datetime")
)

df_trips.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+------------------+----------+----+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|     tripID|     trip_duration|       day|hour|day_of_week|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+------------------+----------+----+-----------+
|       1|

In [77]:
# Busiest day of the week 
df_trips.groupBy("day_of_week").count() \
    .groupBy("day_of_week") \
    .agg(avg("count").alias("average_trips")) \
    .orderBy("average_trips", ascending=False) \
    .show(1)

+-----------+-------------+
|day_of_week|average_trips|
+-----------+-------------+
|          5|    1357043.0|
+-----------+-------------+
only showing top 1 row


In [78]:
# Slowest day of the week 
df_trips.groupBy("day_of_week").count() \
    .groupBy("day_of_week") \
    .agg(avg("count").alias("average_trips")) \
    .orderBy("average_trips", ascending=True) \
    .show(1)

+-----------+-------------+
|day_of_week|average_trips|
+-----------+-------------+
|          1|     859905.0|
+-----------+-------------+
only showing top 1 row


In [79]:
# Does trip distance or num passangers affect tip amount
from pyspark.sql.functions import avg

# Average tip by trip distannce
df_trips.groupBy("trip_distance") \
    .agg(avg("tip_amount").alias("average_tip")) \
    .orderBy("trip_distance") \
    .show()

+-------------+------------------+
|trip_distance|       average_tip|
+-------------+------------------+
|          0.0|2.9063733231680193|
|         0.01| 1.507821242881903|
|         0.02|1.2754726930320155|
|         0.03|1.4499184261036482|
|         0.04|1.5668838352796568|
|         0.05| 1.729612129760226|
|         0.06|1.8371002485501255|
|         0.07|1.4241458910433986|
|         0.08|1.3678550148957302|
|         0.09|1.8145353982300887|
|          0.1|0.9032361666451706|
|         0.11|1.4208571428571426|
|         0.12| 1.174978902953587|
|         0.13|1.0578539823008855|
|         0.14| 1.206410256410256|
|         0.15|0.9320209973753291|
|         0.16|0.8956394849785412|
|         0.17|0.9528304947283054|
|         0.18|0.7876534052596096|
|         0.19|0.7550280504908838|
+-------------+------------------+
only showing top 20 rows


In [80]:
from pyspark.sql.functions import when, avg, count

df_trips = df_trips.withColumn(
    "distance_bucket",
    when(df_trips.trip_distance < 2, "0-2 miles")
    .when(df_trips.trip_distance < 5, "2-5 miles")
    .when(df_trips.trip_distance < 10, "5-10 miles")
    .otherwise("10+ miles")
)

df_trips.groupBy("distance_bucket") \
    .agg(
        avg("tip_amount").alias("average_tip"),
        count("*").alias("number_of_trips")
    ) \
    .show()

+---------------+------------------+---------------+
|distance_bucket|       average_tip|number_of_trips|
+---------------+------------------+---------------+
|     5-10 miles|3.4699004511456972|         585843|
|      2-5 miles|1.9824044025647547|        1919063|
|      10+ miles| 6.219343545352378|         441310|
|      0-2 miles|1.1435660673705301|        4750401|
+---------------+------------------+---------------+



There appears to be a positive relationship between trip distance and the average tip amount. The average tip increases from $1.14 for trips shorter than 2 miles to $1.98 for trips between 2 and 5 miles, $3.47 for trips between 5 and 10 miles, and $6.22 for trips longer than 10 miles. This suggests that longer trips are associated with higher tip amounts. However, this does not necessarily mean that trip distance directly causes higher tips, since longer trips generally also have higher fares.

In [81]:
# Avergae tip by passenger count
df_trips.groupBy("passenger_count") \
    .agg(avg("tip_amount").alias("average_tip")) \
    .orderBy("passenger_count") \
    .show()

+---------------+--------------------+
|passenger_count|         average_tip|
+---------------+--------------------+
|           NULL|0.061789899553571406|
|            0.0|  1.7869007761051638|
|            1.0|  1.8283524429075058|
|            2.0|  1.8339324029045228|
|            3.0|  1.7955889568213272|
|            4.0|  1.7027097823846875|
|            5.0|  1.8698681146978595|
|            6.0|  1.8568302035247934|
|            7.0|   6.542631578947368|
|            8.0|   6.480689655172414|
|            9.0|  3.1166666666666667|
+---------------+--------------------+



The relationship between passenger count and tip amount does not appear to be strong for most passenger counts. The average tip is relatively similar for trips with 0 to 6 passengers. Trips with 7 and 8 passengers have higher average tips.

In [82]:
# What was the highest "extra" charge and which trip
df_trips.select("tripID", "extra") \
    .orderBy("extra", ascending=False) \
    .show(1)

+-----------+------+
|     tripID| extra|
+-----------+------+
|60134865627|535.38|
+-----------+------+
only showing top 1 row


The highest extra charge in the dataset is 535.38, corresponding to trip 60134865627.

**Are there any datapoints that seem to be strange/outliers ?**
Several datapoints appear to be unusual or potential outliers. The most obvious example is trip **60135616235**, which has a trip distance of **831.8 miles**, which is extremely high for a New York City taxi trip.

Another clear anomaly is trip **60130745328**, which has a trip duration of **-84,280.5 minutes**. A negative duration is impossible because the drop-off time occurs before the pick-up time, suggesting an error in the data.

Trip **60129610411** is also unusual because its duration is approximately **43,648 minutes**, or about 30 days, which is highly unlikely for a taxi trip.

The extra value also contains a potential outlier: trip **60134865627** has an extra charge of **535.38**, which is unusually high. In addition, some trips have a distance of **0 miles**, including a trip with a fare of **52.00**, which may indicate an unusual transaction or a data quality issue.


### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [83]:
# Load in the taxi zone lookup
download_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
taxi_zone_lookup_data = "taxi_zone_lookup.csv"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(taxi_zone_lookup_data, "wb") as f:
        f.write(response.content)

In [84]:
# create the dataframe
df_taxi_zone_lookup = spark.read.csv(taxi_zone_lookup_data, header=True, inferSchema=True)

In [85]:
df_taxi_zone_lookup.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [86]:
# Show the dataframe
df_taxi_zone_lookup.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [87]:
# 1st join
df_zone_pu = df_taxi_zone_lookup \
    .withColumnRenamed("LocationID", "PU_LocationID") \
    .withColumnRenamed("Borough", "PU_Borough") \
    .withColumnRenamed("Zone", "PU_Zone") \
    .withColumnRenamed("service_zone", "PU_service_zone")


print(df_zone_pu.printSchema())

df_resultat = df_trips.join(df_zone_pu, df_trips["PULocationID"] == df_zone_pu["PU_LocationID"], "inner").drop("PU_LocationID")

print(df_resultat.printSchema())
print(len(df_resultat.columns))


root
 |-- PU_LocationID: integer (nullable = true)
 |-- PU_Borough: string (nullable = true)
 |-- PU_Zone: string (nullable = true)
 |-- PU_service_zone: string (nullable = true)

None
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullab

In [88]:
# 2nd join
df_zone_do = df_taxi_zone_lookup \
    .withColumnRenamed("LocationID", "DO_LocationID") \
    .withColumnRenamed("Borough", "DO_Borough") \
    .withColumnRenamed("Zone", "DO_Zone") \
    .withColumnRenamed("service_zone", "DO_service_zone")

print(df_zone_do.printSchema())

df_trips_with_zones = df_resultat.join(df_zone_do, df_resultat["DOLocationID"] == df_zone_do["DO_LocationID"], "inner").drop("DO_LocationID")

print(df_trips_with_zones.printSchema())
print(len(df_trips_with_zones.columns))


root
 |-- DO_LocationID: integer (nullable = true)
 |-- DO_Borough: string (nullable = true)
 |-- DO_Zone: string (nullable = true)
 |-- DO_service_zone: string (nullable = true)

None
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullab

In [89]:
df_trips_with_zones.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+------------------+----------+----+-----------+---------------+----------+--------------------+---------------+----------+--------------------+---------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|     tripID|     trip_duration|       day|hour|day_of_week|distance_bucket|PU_Borough|             PU_Zone|PU_service_zone|DO_Borough|             DO_Zone|DO_service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+---

In [90]:
df_trips_with_zones.count()

7696617

In [91]:
# Which borough had most pickups? dropoffs?
borough_pickups = df_trips_with_zones.groupBy("PU_Borough").count().orderBy("count", ascending=False)
borough_dropoffs = df_trips_with_zones.groupBy("DO_Borough").count().orderBy("count", ascending=False)

print("Borough with most pickups:")
borough_pickups.show(2)
print("\nBorough with most dropoffs:")
borough_dropoffs.show(2)

Borough with most pickups:
+----------+-------+
|PU_Borough|  count|
+----------+-------+
| Manhattan|6950965|
|    Queens| 471173|
+----------+-------+
only showing top 2 rows

Borough with most dropoffs:
+----------+-------+
|DO_Borough|  count|
+----------+-------+
| Manhattan|6817355|
|    Queens| 340972|
+----------+-------+
only showing top 2 rows


In [100]:
# What are the busy/slow times by borough 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_hour = df_trips_with_zones.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
hour_counts = df_hour.groupBy("PU_Borough", "pickup_hour").count()

w_max = Window.partitionBy("PU_Borough").orderBy(F.desc("count"))
w_min = Window.partitionBy("PU_Borough").orderBy(F.asc("count"))

busiest_hour = hour_counts.withColumn("rank", F.row_number().over(w_max)).filter("rank == 1").drop("rank")
slowest_hour = hour_counts.withColumn("rank", F.row_number().over(w_min)).filter("rank == 1").drop("rank")
busiest_hour.show()
slowest_hour.show()

+-------------+-----------+------+
|   PU_Borough|pickup_hour| count|
+-------------+-----------+------+
|        Bronx|          7|  1803|
|     Brooklyn|          8|  6935|
|          EWR|         15|    54|
|    Manhattan|         18|471539|
|          N/A|         19|   214|
|       Queens|         16| 29885|
|Staten Island|          8|    36|
|      Unknown|         18| 10751|
+-------------+-----------+------+

+-------------+-----------+-----+
|   PU_Borough|pickup_hour|count|
+-------------+-----------+-----+
|        Bronx|          3|  225|
|     Brooklyn|          3| 1919|
|          EWR|         23|    1|
|    Manhattan|          4|53447|
|          N/A|          6|   88|
|       Queens|          3| 3085|
|Staten Island|          1|    3|
|      Unknown|          4| 1465|
+-------------+-----------+-----+



In [101]:
# What are the busiest days of the week by borough?
df_dow = df_trips_with_zones.withColumn("pickup_dow", F.dayofweek("tpep_pickup_datetime"))
dow_counts = df_dow.groupBy("PU_Borough", "pickup_dow").count()
busiest_dow = dow_counts.withColumn(
    "rank", F.row_number().over(Window.partitionBy("PU_Borough").orderBy(F.desc("count")))
).filter("rank == 1").drop("rank")
busiest_dow.show()

+-------------+----------+-------+
|   PU_Borough|pickup_dow|  count|
+-------------+----------+-------+
|        Bronx|         5|   3121|
|     Brooklyn|         3|  15779|
|          EWR|         4|     83|
|    Manhattan|         5|1229554|
|          N/A|         3|    703|
|       Queens|         5|  78972|
|Staten Island|         6|     64|
|      Unknown|         5|  28929|
+-------------+----------+-------+



In [102]:
# What is the average trip distance by borough?
df_trips_with_zones.groupBy("PU_Borough").agg(F.avg("trip_distance").alias("avg_distance")).show()


+-------------+------------------+
|   PU_Borough|      avg_distance|
+-------------+------------------+
|       Queens|11.283218499361993|
|          EWR| 2.641098654708519|
|      Unknown| 2.415464130400774|
|     Brooklyn| 4.787677275447492|
|Staten Island|12.503601108033246|
|          N/A| 3.193850899742941|
|    Manhattan|2.2286693358402596|
|        Bronx| 7.233194552098303|
+-------------+------------------+



In [103]:
# What is the average trip fare by borough?
df_trips_with_zones.groupBy("PU_Borough").agg(F.avg("fare_amount").alias("avg_fare")).show()


+-------------+------------------+
|   PU_Borough|          avg_fare|
+-------------+------------------+
|       Queens| 35.14462651722029|
|          EWR| 76.24024663677126|
|      Unknown|14.944423051653523|
|     Brooklyn|18.649132800172286|
|Staten Island|45.289861495844896|
|          N/A|  59.5731593830335|
|    Manhattan|10.792468572351568|
|        Bronx| 26.26890543682963|
+-------------+------------------+



In [104]:
# What are the highest and lowest fare amounts for a trip, and which borough is associated with each?
df_trips_with_zones.orderBy(F.desc("fare_amount")).select("PU_Borough", "fare_amount").show(1)
df_trips_with_zones.orderBy(F.asc("fare_amount")).select("PU_Borough", "fare_amount").show(1)


+----------+-----------+
|PU_Borough|fare_amount|
+----------+-----------+
| Manhattan|  623259.86|
+----------+-----------+
only showing top 1 row
+----------+-----------+
|PU_Borough|fare_amount|
+----------+-----------+
|    Queens|     -362.0|
+----------+-----------+
only showing top 1 row


In [97]:
# Load the dataset from the most recently available January, and determine if there are any changes to the average metrics.
# set dl url for January 2026 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2026_trip_data = "yellow_tripdata_2026-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2026_trip_data, "wb") as f:
        f.write(response.content)


In [98]:
# create the dataframe
df_trips_2026 = spark.read.parquet(jan_2026_trip_data)

In [99]:
# Show the dataframe
df_trips_2026.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         1|                 N|         239|    

In [105]:
df_trips_recent_zones = df_trips_2026 \
    .join(df_zone_pu, df_trips_2026["PULocationID"] == df_zone_pu["PU_LocationID"], "inner").drop("PU_LocationID") \
    .join(df_zone_do, df_trips_2026["DOLocationID"] == df_zone_do["DO_LocationID"], "inner").drop("DO_LocationID")

df_trips_with_zones.groupBy("PU_Borough").agg(F.avg("trip_distance").alias("avg_distance_old"), F.avg("fare_amount").alias("avg_fare_old")) \
    .join(
        df_trips_recent_zones.groupBy("PU_Borough").agg(F.avg("trip_distance").alias("avg_distance_new"), F.avg("fare_amount").alias("avg_fare_new")),
        "PU_Borough"
    ).show()

+-------------+------------------+------------------+-------------------+------------------+
|   PU_Borough|  avg_distance_old|      avg_fare_old|   avg_distance_new|      avg_fare_new|
+-------------+------------------+------------------+-------------------+------------------+
|       Queens|11.283218499361993| 35.14462651722029| 13.779063179690827| 45.22028597031559|
|          EWR| 2.641098654708519| 76.24024663677126|0.47888888888888886| 89.72720164609055|
|      Unknown| 2.415464130400774|14.944423051653523| 3.3269961284445464|20.784709633340906|
|     Brooklyn| 4.787677275447492|18.649132800172286| 16.840026980496987|30.850515006292362|
|Staten Island|12.503601108033246|45.289861495844896|  8.225288065843621|41.413065843621396|
|          N/A| 3.193850899742941|  59.5731593830335|  4.218869395711498| 87.74134502923974|
|    Manhattan|2.2286693358402596|10.792468572351568|   5.09188978078639| 17.48394860791505|
|        Bronx| 7.233194552098303| 26.26890543682963| 12.2838808762132

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing